<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/giovanni/MOD-1/notebooks/04_age_prediction_eda_classification_holdout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Age Prediction: data preparation, Exploratory Data Analysis (EDA), and classification with hold-out & nested hold-out

## Data Preparation and Exploratory Data Analysis (EDA)

Import libraries

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Loading and showing input (features) and output data (desired output)

In [ ]:
df = pd.read_excel('https://raw.githubusercontent.com/sdiciotti/Age-Prediction-Demo/main/NKI2_data.xlsx')
print (type(df))

Print the size of the dataframe

In [ ]:
df.shape #73,35

Print the dataframe

In [ ]:
print(df)

Check the features and output variables name

In [ ]:
features = list(df.columns)
type(features)

In [ ]:
print(features)

Plotting some data

In [ ]:
# Plot true vs predicted values to visualize model performance
plt.scatter(df['Age'],df['cortex_CT'])
plt.xlabel('Age (years)')
plt.ylabel('cortex CT (mm)') #SAME AS OTHER AGE PREDICTION

In [ ]:
print("Dataframe shape before NaN removal:", df.shape)

Dataframe shape before NaN removal: (73, 35)


Removing missing data

In [ ]:
df.dropna(axis=0, how='any', inplace=True) #AFTER REMOVAL (72,35)
print("Dataframe shape after NaN removal:", df.shape)

Preparing the design matrix X and the desired output y

In [ ]:
X = df.iloc[:,2::]
y = df['Age']
print (type(X))
print (type(y))
print (X.shape)
print (y.shape)

A quick quality control

In [ ]:
X.head(6)

In [ ]:
print('The whole dataset contains ' + str(X.shape[0]) + ' subjects')
print('The age prediction will be performed using ' + str(X.shape[1]) + ' MRI-derived features')

The whole dataset contains 72 subjects
The age prediction will be performed using 33 MRI-derived features


## Preparing data for a classification task

In [ ]:
#HERE THE TASK IS CLASSIFICATION: not predicting a continuos number, but predicting which age group subjects belong to:
#I need to convert age continuous variable into discrete categories (binarization/discretization) and consider classification models

Define a function to binarize age

In [ ]:
def categorize_age(age): #I need discrete clas labels (binary), 0 for child 1 if "adults"
    if age <= 11:
        return 0
    elif age >= 12: #chosen this as passage from childhood to adolescence
        return 1

Binarizing age to obtain two classes

In [ ]:
df['Age_Category'] = df['Age'].apply(categorize_age)
#df['Age_Category'] = df['Age_Category'].astype(int)
# Count the occurrences of each unique value in the 'Age_Category' column
age_category_counts = df['Age_Category'].value_counts()
# Display the counts; checking for class imbalance
print("Number of rows with Age_Category equal to 0:", age_category_counts[0])
print("Number of rows with Age_Category equal to 1:", age_category_counts[1])

Number of rows with Age_Category equal to 0: 32
Number of rows with Age_Category equal to 1: 40


A quick quality control

In [ ]:
df.tail(10) #visuality check (end of dataset)

Extracting "Age_category" and removing the "Age", "Sex" columns from the data

In [ ]:
y = df['Age_Category'] #target is now binary
X = df.drop(columns=['Age_Category','Age','Sex'])
#in feature matrix are removed three columns, age category because it is the output, age because it's what age category is derived from
#and sex is also excluded to focus just on brain derived measurements --> AVOIDING DATA LEAKAGE, I need to give to the model just MRI features
#and understand if from those alone is possible to distinguish two age groups, don't need other info that "spoiler" or confound the process

# Convert the 'Age_Category' column to integer type
#df['Age_Category'] = df['Age_Category'].astype(int)
print('The whole dataset contains ' + str(X.shape[0]) + ' subjects')
print('The age prediction will be performed using ' + str(X.shape[1]) + ' MRI-derived features')

The whole dataset contains 72 subjects
The age prediction will be performed using 33 MRI-derived features


A quick quality control

In [ ]:
X.tail(10)

In [ ]:
y.tail(10)

## Classification task

### Logistic regression using a holdout scheme

In [ ]:
#LOGISTIC REGRESSION is a classification algorithm --> it models the probability that a subject belongs to class 1, by
#fitting a linear combination of features and putting the result through a sigmoid function (output btw 0 and 1)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# Setting the seed of the random generator
SEED = 42
# Size of the samples in the test set: e.g., 0.1, means that the test set is composed of 10% of the samples of the entire dataset
test_size = 0.1

# Creating the splitter
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=SEED)

In [ ]:
print (X_train.shape, y_train.shape)
print (X_test.shape, y_test.shape) #splitting same dimensions as other exercise

(64, 33) (64,)
(8, 33) (8,)


In [ ]:
from sklearn.metrics import roc_auc_score

clf_holdout = LogisticRegression(max_iter=1000) #enough iterations to converge
clf_holdout.fit(X_train, y_train) #only training data
y_pred_proba_test = clf_holdout.predict_proba(X_test)[:, 1]
y_pred_proba_train = clf_holdout.predict_proba(X_train)[:, 1]
#for every subjects, kept only probability of beloning to class 1 (adolescent+) --> MODEL'S CONFIDENCE SCORE
auc_train = roc_auc_score(y_train, y_pred_proba_train)
auc_test = roc_auc_score(y_test, y_pred_proba_test)
#AUC measured how well model's predicted probabilities rank subjects --> remembering that it is THRESHOLD INDEPENDENT AND INSENSITIVE TO CLASS IMBALANCE
#IF both AUC ARE very high (like in this case) MODELS predicts very well --> I need to consider test set in particular is bery narrow , check maybe for wider datasets

print("Average AUC training set:", auc_train)
print("Average AUC test set:", auc_test)

Average AUC training set: 0.968627450980392
Average AUC test set: 1.0


### Logistic Regression with hyperparameter C (Complexity) using a nested holdout scheme

In [ ]:
from sklearn.model_selection import train_test_split
# Setting the seed of the random generator
SEED = 42

# Size of the samples in the test set: e.g., 0.1, means that the test set is composed of 10% of the samples of the entire dataset
test_size = 0.1

# Size of the samples in the validation set: e.g., 0.1, means that the test set is composed of 10% of the samples of the development set
val_size = 0.1

# Outer hold-out
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=test_size, random_state=SEED)
# Print the size of the development and test sets
print ("The development set size is", "X:", X_dev.shape, "y:", y_dev.shape)
print ("The test set size is", "X:", X_test.shape, "y:", y_test.shape)
# Inner hold-out
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=val_size, random_state=SEED)
# Print the size of the training and validation sets
print ("\nThe development set size is", "X:", X_train.shape, "y:", y_train.shape)
print ("The test set size is", "X:", X_val.shape, "y:", y_val.shape)
print()

The development set size is X: (64, 33) y: (64,)
The test set size is X: (8, 33) y: (8,)

The development set size is X: (57, 33) y: (57,)
The test set size is X: (7, 33) y: (7,)



In [ ]:
AUC = []
C_list = [0.1, 1, 10, 100] #small c strong regularization vs large c gives a weak regularization
for c in C_list:
    clf = LogisticRegression(max_iter=1000, C=c).fit(X_train, y_train)
    y_pred_proba_val = clf.predict_proba(X_val)[:, 1]
    auc_val = roc_auc_score(y_val, y_pred_proba_val) #this because AUC threshold independent
    print("Average AUC val set:", auc_val)
    AUC.append (auc_val)
bestC_index = np.argmax(AUC)
bestC = C_list[bestC_index]
print ("\nBest C: ", bestC, "AUC", AUC[bestC_index])
# Re-train the SVR model in the development test using the bestC hyperparameter
clf = LogisticRegression(max_iter=1000,C=bestC).fit(X_dev, y_dev)# refit on full development set X_dev = train + val
# Apply the model to the test set and compute the MAE
y_pred_test = clf.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_pred_proba_test)
print("Average AUC test set:", auc_test)

Average AUC val set: 0.9166666666666667
Average AUC val set: 0.9166666666666667
Average AUC val set: 0.9166666666666667
Average AUC val set: 0.75

Best C:  0.1 AUC 0.9166666666666667
Average AUC test set: 1.0
